In [1]:
import torch
from compressai.zoo import cheng2020_anchor
from torchsummary import summary

In [2]:
# Si tenemos disponible GPU, lo usamos
# Chequeamos si tenemos disponible GPU (CUDA)
if torch.cuda.is_available():
    device = "cuda"
# Chequeamos si tenemos disponible aceleración por hardware en un chip de Apple (MPS)
elif torch.backends.mps.is_available():
    device = "mps"
# Por defecto usamos CPU
else:
    device = "cpu"

### Carga del modelo preentrenado

In [3]:
# Cargo el modelo preentrenado
model = cheng2020_anchor(quality=1, pretrained=True).to(device)

In [4]:
# Imprimo la arquitectura del modelo
print(model)

Cheng2020Anchor(
  (entropy_bottleneck): EntropyBottleneck(
    (likelihood_lower_bound): LowerBound()
  )
  (g_a): Sequential(
    (0): ResidualBlockWithStride(
      (conv1): Conv2d(3, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (leaky_relu): LeakyReLU(negative_slope=0.01, inplace=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (gdn): GDN(
        (beta_reparam): NonNegativeParametrizer(
          (lower_bound): LowerBound()
        )
        (gamma_reparam): NonNegativeParametrizer(
          (lower_bound): LowerBound()
        )
      )
      (skip): Conv2d(3, 128, kernel_size=(1, 1), stride=(2, 2))
    )
    (1): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (leaky_relu): LeakyReLU(negative_slope=0.01, inplace=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (2): ResidualBlockWithStride(
      (conv1): Conv2d(1

In [5]:
# Asumiendo que las imágenes de entrada tienen 3 canales y son de 256x256 como lo expresa en el repositorio
summary(model, (3, 256, 256), device='cuda')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1        [-1, 128, 128, 128]           3,584
         LeakyReLU-2        [-1, 128, 128, 128]               0
            Conv2d-3        [-1, 128, 128, 128]         147,584
        LowerBound-4                       [-1]               0
NonNegativeParametrizer-5                       [-1]               0
        LowerBound-6                  [-1, 128]               0
NonNegativeParametrizer-7                  [-1, 128]               0
               GDN-8        [-1, 128, 128, 128]               0
            Conv2d-9        [-1, 128, 128, 128]             512
ResidualBlockWithStride-10        [-1, 128, 128, 128]               0
           Conv2d-11        [-1, 128, 128, 128]         147,584
        LeakyReLU-12        [-1, 128, 128, 128]               0
           Conv2d-13        [-1, 128, 128, 128]         147,584
        LeakyReLU-14   

### Verificación del modelo

In [6]:
# Crearw un tensor de ejemplo con una imagen de 3 canales de 256x256
# Valores aleatorios solo para demostrar; en la práctica, usaremos una imagen real
x = torch.randn(1, 3, 256, 256).to(device)  # Batch size de 1, 3 canales, 256x256 dimensiones

In [7]:
# Paso el tensor al modelo
output = model(x)

In [8]:
# Verifico si la salida es un diccionario y mostrar las claves
if isinstance(output, dict):
    print("Claves del diccionario de salida:")
    for key in output.keys():
        print(key)
else:
    print("La salida no es un diccionario.")

Claves del diccionario de salida:
x_hat
likelihoods


In [9]:
# Accedo a la imagen reconstruida
x_hat = output['x_hat']

In [10]:
# Accedo a las probabilidades asociadas con la compresión
likelihoods = output['likelihoods']

In [11]:
# Inspeccionamos las formas de estos elementos
print(f"Forma de x_hat: {x_hat.shape}")
print(f"Tipo de dato de x_hat: {x_hat.dtype}")
print(f"Contenido de likelihoods: {likelihoods}")

Forma de x_hat: torch.Size([1, 3, 256, 256])
Tipo de dato de x_hat: torch.float32
Contenido de likelihoods: {'y': tensor([[[[0.2394, 0.2293, 0.2346,  ..., 0.1573, 0.1851, 0.2079],
          [0.2106, 0.2164, 0.1759,  ..., 0.2711, 0.2284, 0.2228],
          [0.2683, 0.2287, 0.1869,  ..., 0.2345, 0.1438, 0.2182],
          ...,
          [0.2514, 0.2533, 0.1925,  ..., 0.1898, 0.2099, 0.2025],
          [0.2701, 0.2126, 0.2499,  ..., 0.2141, 0.2548, 0.1454],
          [0.2462, 0.2303, 0.2274,  ..., 0.1637, 0.2078, 0.1943]],

         [[0.4702, 0.1831, 0.2134,  ..., 0.1423, 0.1261, 0.1693],
          [0.8906, 0.0895, 0.0655,  ..., 0.1170, 0.1665, 0.1616],
          [0.6598, 0.1307, 0.0766,  ..., 0.1583, 0.1567, 0.1202],
          ...,
          [0.5401, 0.0969, 0.1404,  ..., 0.1075, 0.1474, 0.1804],
          [0.6579, 0.0959, 0.1306,  ..., 0.1263, 0.1332, 0.1258],
          [0.5739, 0.0736, 0.1097,  ..., 0.0944, 0.0821, 0.1237]],

         [[0.0314, 0.0242, 0.1890,  ..., 0.0912, 0.1350, 0.1